# NB07：糖分析（酚硫酸法總醣與折射計 Brix）

**食品分析實驗 週次 7｜Nielsen's Food Analysis, 6th ed. 對應章節：碳水化合物 (Carbohydrates, Ch.19)**

## 學習目標 (Learning Objectives)

完成本筆記本後，你應該能夠：

1. 完成酚硫酸法(DuBois)標準曲線的**線性迴歸**（斜率、截距、R²），並畫出殘差圖。
2. 說明「**高 R² 不代表迴歸品質好**」，並用 lack-of-fit（缺適性）檢定驗證這件事。
3. 計算 **LOD = 3.3σ/斜率** 與 **LOQ = 10σ/斜率**，並比較「用迴歸殘差 SD」跟「用空白重複讀值 SD」兩種 σ 的差異。
4. 把未知樣品的吸光值換算成 **g/100 mL 葡萄糖當量**（含稀釋倍數），並估計預測不確定度。
5. 比較柳橙汁的 **Brix** 與**酚硫酸法總醣**，理解兩者測的是不同的東西。
6. 用**質量守恆**驗算楓糖漿「幾份樹液換一份糖漿」的說法。
7. 認識勞動部乙級的**銅還原滴定法**(索摩基/Bertrand)的計算邏輯（概念示範，非實際操作數據）。

## 如何使用本筆記本

- 直接「全部執行」（Run all / 執行階段 → 全部執行）即可跑完整份筆記本，使用內建的**模擬教學資料**。
- `load_data()` 支援三種資料來源：
  1. **本機/雲端硬碟路徑**：`load_data("data/nb07_sample.csv")` 或你自己上傳的檔名。
  2. **已發布的 Google 試算表 CSV 連結**：檔案 → 共用 → 發布到網路 → 選 CSV，把網址傳入 `load_data(那個網址)`。
  3. **在 Colab 上傳檔案**：執行 Setup 區塊中被註解掉的 `google.colab.files.upload()` 程式碼。
  4. 若以上都沒有提供，`load_data()` 會自動退回使用本筆記本內嵌的模擬資料字串，確保筆記本在任何環境都能獨立執行。

> ⚠️ **本筆記本內建的資料是模擬教學資料（simulated teaching data）**，不是真實量測值。標準曲線第 1 重複的數字取自本課 W7 投影片的範例，第 2 重複、空白重複讀值、以及各未知樣品的稀釋設計皆為配合教學另行模擬。做作業時請換成你們班的實際數據。

## 資料格式 (Data Schema)

本週的資料同時涉及「標準曲線的已知濃度」與「未知樣品的稀釋倍數」，因此在共用長格式之外多加兩欄：

| 欄位 | 說明 |
|---|---|
| `group` | 組別（1–5）；`0` 代表全班共用（標準曲線、試劑空白） |
| `student` | 代號 |
| `item` | 量測項目：`std_curve`、`reagent_blank_490nm`、`unknown_*`、`orange_juice_brix` 等 |
| `rep` | 重複次數 |
| `value` | 量測值（吸光值 AU，或 Brix 讀值） |
| `unit` | `AU`（490nm 吸光值）或 `degBrix` |
| `temp_C` | 本週不適用，保留欄位一致，留空 |
| `conc_ug_mL` | **新增欄位**：標準品的已知濃度（µg/mL），只有 `std_curve` 才有值 |
| `dilution_factor` | **新增欄位**：未知樣品的稀釋倍數，只有 `unknown_*` 才有值 |
| `note` | 備註欄 |

## 1. 環境設定 (Setup)

匯入需要的套件，並定義 `load_data()`。**注意**：所有圖表的座標軸標籤與標題一律使用**英文**，避免 Colab 預設字型無法顯示中文而出現方框亂碼（tofu）。文字說明則使用繁體中文。

In [ ]:
import io
import os

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

plt.style.use("ggplot")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.precision", 4)

RNG_SEED = 42
np.random.seed(RNG_SEED)

In [ ]:
# ---- Embedded sample data (fallback) ----------------------------------
# This is the exact content of data/nb07_sample.csv, embedded so the
# notebook is self-contained (works even without the data/ folder present).
SAMPLE_CSV = """group,student,item,rep,value,unit,temp_C,conc_ug_mL,dilution_factor,note
0,class_curve,std_curve,1,0.021,AU,,0.0,,標準曲線第 1 重複，本班的正式檢量線
0,class_curve,std_curve,1,0.233,AU,,20.0,,標準曲線第 1 重複，本班的正式檢量線
0,class_curve,std_curve,1,0.447,AU,,40.0,,標準曲線第 1 重複，本班的正式檢量線
0,class_curve,std_curve,1,0.64,AU,,60.0,,標準曲線第 1 重複，本班的正式檢量線
0,class_curve,std_curve,1,0.847,AU,,80.0,,標準曲線第 1 重複，本班的正式檢量線
0,class_curve,std_curve,1,1.029,AU,,100.0,,標準曲線第 1 重複，本班的正式檢量線
0,class_curve,std_curve,2,0.0227,AU,,0.0,,標準曲線第 2 重複，用於示範 lack-of-fit 檢定（需要重複才能估計純誤差）
0,class_curve,std_curve,2,0.2268,AU,,20.0,,標準曲線第 2 重複，用於示範 lack-of-fit 檢定（需要重複才能估計純誤差）
0,class_curve,std_curve,2,0.4496,AU,,40.0,,標準曲線第 2 重複，用於示範 lack-of-fit 檢定（需要重複才能估計純誤差）
0,class_curve,std_curve,2,0.6532,AU,,60.0,,標準曲線第 2 重複，用於示範 lack-of-fit 檢定（需要重複才能估計純誤差）
0,class_curve,std_curve,2,0.8259,AU,,80.0,,標準曲線第 2 重複，用於示範 lack-of-fit 檢定（需要重複才能估計純誤差）
0,class_curve,std_curve,2,1.0205,AU,,100.0,,標準曲線第 2 重複，用於示範 lack-of-fit 檢定（需要重複才能估計純誤差）
0,class_blank,reagent_blank_490nm,1,0.0205,AU,,,,試劑空白重複讀值，用於估計 blank SD（LOD 的另一種算法）
0,class_blank,reagent_blank_490nm,2,0.0187,AU,,,,試劑空白重複讀值，用於估計 blank SD（LOD 的另一種算法）
0,class_blank,reagent_blank_490nm,3,0.0199,AU,,,,試劑空白重複讀值，用於估計 blank SD（LOD 的另一種算法）
0,class_blank,reagent_blank_490nm,4,0.0166,AU,,,,試劑空白重複讀值，用於估計 blank SD（LOD 的另一種算法）
0,class_blank,reagent_blank_490nm,5,0.0235,AU,,,,試劑空白重複讀值，用於估計 blank SD（LOD 的另一種算法）
0,class_blank,reagent_blank_490nm,6,0.0231,AU,,,,試劑空白重複讀值，用於估計 blank SD（LOD 的另一種算法）
0,class_blank,reagent_blank_490nm,7,0.0203,AU,,,,試劑空白重複讀值，用於估計 blank SD（LOD 的另一種算法）
0,class_blank,reagent_blank_490nm,8,0.0245,AU,,,,試劑空白重複讀值，用於估計 blank SD（LOD 的另一種算法）
1,G1,unknown_sports_drink,1,0.637,AU,,,1000.0,運動飲料，已知標準曲線範例（W7 課堂計算闖關）
2,G2,unknown_orange_juice_phenol,1,0.5109,AU,,,2000.0,柳橙汁，酚硫酸法總醣
2,G2,orange_juice_brix,1,11.5,degBrix,,,,柳橙汁，折射計 Brix 讀值（20°C 對照，未做溫度校正）
3,G3,unknown_maple_syrup,1,0.5378,AU,,,17500.0,楓糖漿，酚硫酸法總醣，稀釋倍數很大(原液濃度高)
4,G4,unknown_winter_melon_tea,1,0.5306,AU,,,2000.0,冬瓜茶，樣品本身顏色深，讀值含顏色干擾(尚未扣除樣品空白)
4,G4,winter_melon_tea_sample_blank,1,0.045,AU,,,2000.0,冬瓜茶樣品本身的顏色空白(同稀釋倍數，不加酚/硫酸或以水代替糖反應)，用於扣除色素干擾
"""

In [ ]:
def load_data(source=None):
    """Load NB07 sugar-analysis data in the shared long-format schema:
    group, student, item, rep, value, unit, temp_C, conc_ug_mL, dilution_factor, note

    Parameters
    ----------
    source : str or None
        - None: try a local "data/nb07_sample.csv" relative path first,
          otherwise fall back to the embedded SAMPLE_CSV string.
        - a local file path: read with pandas.read_csv.
        - an http(s) URL (e.g. a published Google Sheets CSV link): read
          directly with pandas.read_csv.

    Returns
    -------
    pandas.DataFrame
    """
    if source is not None:
        try:
            df = pd.read_csv(source)
        except Exception as e:
            print(f"讀取 {source} 失敗，改用內建模擬資料。錯誤訊息：{e}")
            df = None
    else:
        local_path = os.path.join("data", "nb07_sample.csv")
        if os.path.exists(local_path):
            df = pd.read_csv(local_path)
        else:
            df = pd.read_csv(io.StringIO(SAMPLE_CSV))

    if df is None:
        df = pd.read_csv(io.StringIO(SAMPLE_CSV))

    df["note"] = df["note"].fillna("")
    return df


df = load_data()
print(f"讀入 {len(df)} 筆資料，涵蓋項目：{sorted(df['item'].unique())}")
df.head(10)

### （選用）在 Colab 上傳班級實際資料

若在 Google Colab 中執行、且要換成全班實際量測值，取消下方程式碼的註解後執行即可；在非 Colab 環境（例如本機 Jupyter）執行這個 cell 不會有任何作用。

In [ ]:
# try:
#     from google.colab import files
#     uploaded = files.upload()
#     fname = list(uploaded.keys())[0]
#     df = load_data(fname)
#     print(f"已改用上傳檔案：{fname}")
# except ImportError:
#     print("非 Colab 環境，略過上傳步驟；請改用 load_data('本機路徑或URL')")

## 2. 標準曲線：葡萄糖 0–100 µg/mL（本班正式檢量線, 第 1 重複）

用第 1 重複（本班的正式檢量線）做線性迴歸：$A = m \times C + b$。

In [ ]:
curve1 = df[(df["item"] == "std_curve") & (df["rep"] == 1)].sort_values("conc_ug_mL")
C1 = curve1["conc_ug_mL"].to_numpy()
A1 = curve1["value"].to_numpy()

slope, intercept = np.polyfit(C1, A1, 1)
A1_pred = slope * C1 + intercept
resid1 = A1 - A1_pred

ss_res = np.sum(resid1 ** 2)
ss_tot = np.sum((A1 - A1.mean()) ** 2)
r2 = 1 - ss_res / ss_tot

print(f"A = {slope:.6f} x C + {intercept:.6f}")
print(f"R^2 = {r2:.5f}")
print("殘差 =", np.round(resid1, 4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(C1, A1, color="tab:blue", label="measured")
axes[0].plot(C1, A1_pred, color="tab:orange", label="linear fit")
axes[0].set_xlabel("Glucose concentration (ug/mL)")
axes[0].set_ylabel("Absorbance at 490 nm")
axes[0].set_title("Phenol-sulfuric standard curve")
axes[0].legend()

axes[1].axhline(0, color="grey", linewidth=1)
axes[1].plot(C1, resid1, "o-", color="tab:red")
axes[1].set_xlabel("Glucose concentration (ug/mL)")
axes[1].set_ylabel("Residual (measured - fit)")
axes[1].set_title("Residual plot")

plt.tight_layout()
plt.show()

## 3. R² 迷思：高 R² 不代表線性假設成立

上面的殘差圖如果不是隨機散佈、而是有規律的弧形（例如兩端偏負、中間偏正），代表資料可能有系統性偏離直線的彎曲（例如高濃度略為飽和）——即使 R² 很高，也該懷疑線性假設。

只看 R² 沒辦法「證明」線性沒問題，因為 R² 沒有重複測量、無法把「彎曲造成的誤差(lack-of-fit)」跟「單純的隨機雜訊(pure error)」分開。這裡我們額外做了**第 2 重複**（同樣 6 個濃度再測一次），就可以做一個正式的 **lack-of-fit F 檢定**。

In [ ]:
curve_all = df[df["item"] == "std_curve"].copy()
C_all = curve_all["conc_ug_mL"].to_numpy()
A_all = curve_all["value"].to_numpy()

slope_all, intercept_all = np.polyfit(C_all, A_all, 1)
pred_all = slope_all * C_all + intercept_all
resid_all = A_all - pred_all
SSE = np.sum(resid_all ** 2)

levels = np.unique(C_all)
level_means = curve_all.groupby("conc_ug_mL")["value"].mean()
reps_per_level = curve_all.groupby("conc_ug_mL")["value"].count()

SS_lof = float(sum(
    reps_per_level[c] * (level_means[c] - (slope_all * c + intercept_all)) ** 2
    for c in levels
))
SS_pe = SSE - SS_lof
df_lof = len(levels) - 2       # 2 parameters (slope, intercept)
df_pe = len(A_all) - len(levels)

F_stat = (SS_lof / df_lof) / (SS_pe / df_pe)
p_value = stats.f.sf(F_stat, df_lof, df_pe)

print(f"pooled fit (2 reps): A = {slope_all:.6f} x C + {intercept_all:.6f}")
print(f"SS_lack-of-fit={SS_lof:.6f} (df={df_lof}), SS_pure-error={SS_pe:.6f} (df={df_pe})")
print(f"F = {F_stat:.3f}, p = {p_value:.4f}")
if p_value < 0.05:
    print("p < 0.05 -> 拒絕『資料為直線』的虛無假設，顯示有統計上顯著的 lack-of-fit（即使 R^2 很高）。")
else:
    print("p >= 0.05 -> 沒有足夠證據顯示偏離直線。")

## 4. LOD / LOQ：兩種 σ 的選擇

$$LOD = \frac{3.3\sigma}{\text{斜率}} \qquad LOQ = \frac{10\sigma}{\text{斜率}}$$

$\sigma$ 有兩種常見取法：

- **迴歸殘差標準差 (residual SD)**：反映整條標準曲線的擬合品質，**包含 lack-of-fit**（如果曲線本身有一點彎曲，殘差 SD 會偏大）。
- **空白重複讀值標準差 (blank SD)**：只反映儀器在零濃度附近的雜訊，**不包含曲線形狀的問題**，通常比殘差 SD 小、也比較樂觀。

兩者都「對」，但代表的意義不同；報告 LOD/LOQ 時務必註明用的是哪一種 $\sigma$。

In [ ]:
resid_sd = np.std(resid1, ddof=2)  # regression residual SD, df = n - 2
LOD_resid = 3.3 * resid_sd / slope
LOQ_resid = 10 * resid_sd / slope

blanks = df[df["item"] == "reagent_blank_490nm"]["value"]
blank_sd = blanks.std(ddof=1)
LOD_blank = 3.3 * blank_sd / slope
LOQ_blank = 10 * blank_sd / slope

print(f"[殘差 SD 法]  sigma={resid_sd:.5f} AU -> LOD={LOD_resid:.2f} ug/mL, LOQ={LOQ_resid:.2f} ug/mL")
print(f"[空白 SD 法]  sigma={blank_sd:.5f} AU (n={len(blanks)}) -> LOD={LOD_blank:.2f} ug/mL, LOQ={LOQ_blank:.2f} ug/mL")
print()
print("空白 SD 法算出的 LOD/LOQ 比較小，因為它只看『零濃度附近的雜訊』；"
      "殘差 SD 法把整條曲線的彎曲(lack-of-fit)也算進去，比較保守，"
      "也比較能反映『在整個工作範圍內都適用』的偵測極限。")

## 5. 未知樣品：從吸光值到 g/100 mL 葡萄糖當量

$$C_{\text{反應液}} = \frac{A - b}{m} \ (\mu g/mL) \qquad \text{總醣 (g/100\ mL)} = C_{\text{反應液}} \times \text{稀釋倍數} \times 10^{-4}$$

並用一個簡化的預測不確定度公式估計 $C$ 的標準誤：

$$s_{C_0} = \frac{s_y}{m}\sqrt{\frac{1}{k} + \frac{1}{n} + \frac{(A_0-\bar{A})^2}{m^2 \cdot S_{xx}}}$$

其中 $k$ 為未知樣品的重複讀值次數（本週每個樣品只讀 1 次，$k=1$）、$n$ 為標準點數、$S_{xx}=\sum(C_i-\bar C)^2$。

In [ ]:
Sxx = np.sum((C1 - C1.mean()) ** 2)


def conc_from_absorbance(A_measured, k=1):
    C0 = (A_measured - intercept) / slope
    sC0 = (resid_sd / slope) * np.sqrt(1 / k + 1 / len(C1) + (A_measured - A1.mean()) ** 2 / (slope ** 2 * Sxx))
    return C0, sC0


def total_sugar_g_per_100mL(A_measured, dilution_factor, k=1):
    C0, sC0 = conc_from_absorbance(A_measured, k=k)
    total = C0 * dilution_factor * 1e-4
    total_sd = sC0 * dilution_factor * 1e-4
    return total, total_sd


unknown_items = ["unknown_sports_drink", "unknown_orange_juice_phenol", "unknown_maple_syrup"]
results = []
for item in unknown_items:
    row = df[df["item"] == item].iloc[0]
    total, total_sd = total_sugar_g_per_100mL(row["value"], row["dilution_factor"])
    results.append({"item": item, "A": row["value"], "dilution_factor": row["dilution_factor"],
                     "total_sugar_g_per_100mL": round(total, 3), "approx_SD": round(total_sd, 3)})

unknowns_df = pd.DataFrame(results)
unknowns_df

### 冬瓜茶：先扣顏色空白，再換算

冬瓜茶顏色很深，樣品本身的顏色會疊加在酚硫酸顯色的吸光值上。必須先用**同稀釋倍數、不含反應顯色的樣品空白**扣除背景色，才能換算總醣。

In [ ]:
tea_raw = df.loc[df["item"] == "unknown_winter_melon_tea"].iloc[0]
tea_blank = df.loc[df["item"] == "winter_melon_tea_sample_blank"].iloc[0]

A_tea_corrected = tea_raw["value"] - tea_blank["value"]
tea_total_wrong, _ = total_sugar_g_per_100mL(tea_raw["value"], tea_raw["dilution_factor"])
tea_total_correct, tea_total_sd = total_sugar_g_per_100mL(A_tea_corrected, tea_raw["dilution_factor"])

print(f"原始讀值 A_raw = {tea_raw['value']:.4f}，樣品顏色空白 A_blank = {tea_blank['value']:.4f}")
print(f"扣除顏色空白後 A_corrected = {A_tea_corrected:.4f}")
print(f"若忘記扣除顏色空白 -> 總醣 ≈ {tea_total_wrong:.2f} g/100mL（高估）")
print(f"扣除顏色空白後      -> 總醣 ≈ {tea_total_correct:.2f} ± {tea_total_sd:.2f} g/100mL")

## 6. 柳橙汁：Brix vs 酚硫酸法總醣

Brix 是以折射率換算的**可溶性固形物**，柳橙汁裡除了糖，還有**酸**等其他可溶物也會提高折射率，所以 °Brix 通常會**大於**酚硫酸法測到的純糖當量。

> **單位提醒**：Brix 的定義是 g/100 g（w/w），酚硫酸法算出的是 g/100 mL（w/v）。柳橙汁密度略大於 1（約 1.04–1.05 g/mL），嚴格比較時應先把其中一個單位換算過去；這裡先直接比較兩個數字的量級，重點是理解「Brix 包含酸、酚硫酸法只對醣類顯色」這個差異的方向，而不是要求兩者在數值上完全可比。

> **溫度提醒**：折射計通常以 20°C 為對照溫度，現代儀器多有自動溫度補償(ATC)；若沒有 ATC，應盡量在建議溫度範圍內操作，而不是套用一個記不清楚的校正公式。

In [ ]:
oj_brix = df.loc[df["item"] == "orange_juice_brix", "value"].iloc[0]
oj_row = df.loc[df["item"] == "unknown_orange_juice_phenol"].iloc[0]
oj_total_sugar, oj_total_sd = total_sugar_g_per_100mL(oj_row["value"], oj_row["dilution_factor"])

gap = oj_brix - oj_total_sugar
print(f"柳橙汁 Brix = {oj_brix:.1f} degBrix")
print(f"柳橙汁酚硫酸法總醣 = {oj_total_sugar:.2f} ± {oj_total_sd:.2f} g/100mL")
print(f"差距 = {gap:.2f}（Brix 把酸與其他可溶性固形物也算進去，所以數值較大）")

## 7. 楓糖漿質量守恆：「40 加侖樹液 = 1 加侖糖漿」是真的嗎？

楓糖漿樹液約 **2% (w/w)** 糖度，煮成糖漿約 **66% (w/w)**。

$$\text{質量比} = \frac{66\%}{2\%} = 33 : 1$$

但工廠說的「40：1」通常指**體積比**，要再乘上密度校正（樹液密度 ≈1.008 g/mL，糖漿密度 ≈1.33 g/mL）：

$$\text{體積比} = \text{質量比} \times \frac{\rho_{\text{糖漿}}}{\rho_{\text{樹液}}}$$

另外有一個經驗法則 **Rule of 86**：所需樹液加侖數 ≈ 86 / 樹液的 °Brix（此處以樹液糖度%近似 °Brix）。

In [ ]:
SAP_PCT, SYRUP_PCT = 0.02, 0.66            # % w/w (as fractions), from the opening-video story
DENS_SAP, DENS_SYRUP = 1.008, 1.33         # g/mL

mass_ratio = SYRUP_PCT / SAP_PCT
vol_ratio = mass_ratio * (DENS_SYRUP / DENS_SAP)
rule86 = 86.0 / (SAP_PCT * 100)
implied_sap_pct_for_40 = SYRUP_PCT * DENS_SYRUP / DENS_SAP / 40.0 * 100

print(f"樹液 {SAP_PCT*100:.0f}%、糖漿 {SYRUP_PCT*100:.0f}% -> 質量比 = {mass_ratio:.1f} : 1")
print(f"體積比(密度校正後) = {mass_ratio:.1f} x ({DENS_SYRUP}/{DENS_SAP}) ≈ {vol_ratio:.1f} : 1")
print(f"Rule of 86： 86/{SAP_PCT*100:.0f} = {rule86:.1f} : 1  (與上面的體積比一致)")
print(f"若體積比要剛好等於工廠說的『40：1』，樹液糖度大約要 {implied_sap_pct_for_40:.2f}%"
      "——量級吻合，這個說法禁得起質量平衡的驗算。")

## 8. 乙級方法概念：銅還原滴定法 (索摩基 / Bertrand)

**本堂課不操作**，這裡只示範核心的化學計量概念，方便你理解勞動部乙級考試的計算題型。

Bertrand 法的邏輯：還原糖把 $Cu^{2+}$ 還原成 $Cu_2O$ 沉澱 → $Cu_2O$ 溶於硫酸鐵 $Fe_2(SO_4)_3$，把 $Fe^{3+}$ 還原成 $Fe^{2+}$（$Cu^+ \to Cu^{2+}$ 放出 1 個電子，$Fe^{3+} \to Fe^{2+}$ 得到 1 個電子，兩者是 1:1 的氧化還原當量關係）→ 再用 $KMnO_4$ 滴定生成的 $Fe^{2+}$。

$$\text{meq } Cu = \text{meq } Fe^{2+} = V_{KMnO_4}(\text{mL}) \times N_{KMnO_4}(\text{eq/L})$$
$$mg\ Cu = \text{meq } Cu \times 63.55$$

再由 mg Cu 查 Bertrand 表換算成還原糖含量（表格本身是實驗室方法規定的經驗對照表，此處不重現，只示範前段的滴定計量）。下面用一組**通用示意數字**示範前半段計算（非本班實測，僅供理解計算邏輯）：

In [ ]:
V_KMnO4, N_KMnO4 = 15.20, 0.1000  # mL, eq/L -- generic illustrative numbers

meq_Cu = V_KMnO4 * N_KMnO4
mg_Cu = meq_Cu * 63.55

print(f"V(KMnO4) = {V_KMnO4} mL, N(KMnO4) = {N_KMnO4} eq/L")
print(f"meq Cu = meq Fe2+ = {meq_Cu:.3f} meq")
print(f"mg Cu = {meq_Cu:.3f} x 63.55 = {mg_Cu:.2f} mg")
print("下一步：查 Bertrand 表，用 mg Cu 對應到還原糖 mg 數（表格為方法規定值，此處不重現）。")

## 9. 練習題 (Exercises)

請在每題下方的空白程式碼區塊中作答（可以用文字 print 說明，也可以搭配計算）。

### 問題 1

如果冬瓜茶忘記扣除自身顏色的樣品空白，總醣會被高估還是低估？請用第 5 節 `A_raw` 與 `A_sample_blank` 的數字實際計算高估的量（g/100mL 與相對%）。

In [ ]:
# TODO: 請在這裡作答

### 問題 2

第 3 節的 lack-of-fit F 檢定得到 p 值多少？這個結果對「我們能不能信任這條標準曲線」有什麼實務上的建議（例如：要不要改用二次曲線、要不要縮小工作範圍）？

In [ ]:
# TODO: 請在這裡作答

### 問題 3

如果改用 DF=1000（而非 2000）稀釋柳橙汁樣品，換算出的吸光值大約會是多少？還落在標準曲線 0–100 µg/mL 的範圍內嗎？為什麼選對稀釋倍數很重要？

In [ ]:
# TODO: 請在這裡作答

### 問題 4

第 4 節算出「殘差 SD 法」跟「空白 SD 法」兩種 LOD。如果你要寫進正式報告，你會選哪一種？為什麼？

In [ ]:
# TODO: 請在這裡作答

### 問題 5

如果某批楓樹液的糖度實測是 2.2%（而不是開場影片假設的 2%），Rule of 86 與密度校正後的體積比各會變成多少？兩者算出來還吻合嗎？

In [ ]:
# TODO: 請在這裡作答

## 10. 匯出結果 (Export)

把標準曲線參數、LOD/LOQ、以及各未知樣品的總醣結果存成 CSV。若在 Colab 環境執行，會另外觸發瀏覽器下載；在本機執行則只會存檔，不會嘗試下載。

In [ ]:
summary_rows = [
    {"item": "std_curve_slope", "value": slope},
    {"item": "std_curve_intercept", "value": intercept},
    {"item": "std_curve_R2", "value": r2},
    {"item": "LOD_resid_sd_ug_per_mL", "value": LOD_resid},
    {"item": "LOQ_resid_sd_ug_per_mL", "value": LOQ_resid},
    {"item": "LOD_blank_sd_ug_per_mL", "value": LOD_blank},
    {"item": "LOQ_blank_sd_ug_per_mL", "value": LOQ_blank},
    {"item": "lack_of_fit_F", "value": F_stat},
    {"item": "lack_of_fit_p", "value": p_value},
]
summary_df = pd.DataFrame(summary_rows)
results_df = pd.concat([summary_df, unknowns_df.rename(columns={"total_sugar_g_per_100mL": "value"})],
                        ignore_index=True, sort=False)

output_path = "nb07_student_results.csv"
results_df.to_csv(output_path, index=False)
print(f"已儲存：{output_path}")

try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print("非 Colab 環境，略過自動下載（檔案已存在本機/雲端硬碟工作目錄）。")